# Limpeza de Dados — Diagnóstico Inicial

Este notebook **não limpa nada ainda**. O objetivo é apenas enxergar os problemas existentes no arquivo `dados/vendas.csv` antes de qualquer tratamento.

## 1. Carregar os dados

In [20]:
import pandas as pd

# O parâmetro sep=',' indica que o separador de colunas é vírgula (padrão de CSV)
# dtype=str carrega tudo como texto por enquanto — assim nada é convertido automaticamente
# e conseguimos ver os valores exatamente como estão no arquivo
df = pd.read_csv('../dados/vendas.csv', dtype=str)

print(f"Linhas: {df.shape[0]}  |  Colunas: {df.shape[1]}")


Linhas: 100  |  Colunas: 9


## 2. Primeiras linhas

`df.head()` mostra as 5 primeiras linhas — serve para ter uma ideia rápida da estrutura e dos valores.

In [21]:
df.head(10)


,id_venda,data_venda,produto,categoria,quantidade,preco_unitario,vendedor,regiao,status
0,1,2024-01-03,Notebook,Eletrônicos,1,"3.500,00",Carlos Silva,Sudeste,concluída
1,2,05/01/2024,mouse,Periféricos,2,"85,50",Ana Lima,Sul,concluída
2,3,2024-01-08,TECLADO,Periféricos,1,"220,00",João Costa,Nordeste,concluída
3,4,09-01-2024,Monitor,Eletrônicos,NaN,"1.200,00",Carlos Silva,Sudeste,concluída
4,5,2024-01-11,notebook,Eletrônicos,2,"3.500,00",NaN,Sul,concluída
5,6,12/01/2024,Cadeira Gamer,Móveis,1,"890,00",Ana Lima,Sudeste,cancelada
6,7,2024-01-15,MOUSE,Periféricos,3,"85,50",Pedro Rocha,Centro-Oeste,concluída
7,8,18-01-2024,teclado,Periféricos,NaN,NaN,Carlos Silva,Nordeste,pendente
8,9,2024-01-20,Monitor,Eletrônicos,1,"1.200,00",João Costa,Sul,concluída
9,10,22/01/2024,NOTEBOOK,Eletrônicos,1,"3.500,00",Ana Lima,Sudeste,concluída


## 3. Visão geral das colunas

`df.info()` mostra o tipo de cada coluna e quantos valores não-nulos existem em cada uma.  
Como carregamos tudo como `str`, todas as colunas aparecem como `object` — o importante aqui é ver se alguma coluna tem menos de 100 entradas (o que indicaria valores faltantes).

In [22]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_venda        100 non-null    object
 1   data_venda      100 non-null    object
 2   produto         100 non-null    object
 3   categoria       100 non-null    object
 4   quantidade      80 non-null     object
 5   preco_unitario  98 non-null     object
 6   vendedor        85 non-null     object
 7   regiao          100 non-null    object
 8   status          100 non-null    object
dtypes: object(9)
memory usage: 7.2+ KB


## 4. Valores faltantes por coluna

`df.isnull().sum()` conta quantas células estão vazias em cada coluna.  
Células vazias no CSV viram `NaN` (Not a Number) no pandas — é assim que ele representa "ausência de valor".

In [23]:
valores_faltantes = df.isnull().sum()

# Filtra só as colunas que têm pelo menos 1 valor faltante
print("Colunas com valores faltantes:")
print(valores_faltantes[valores_faltantes > 0])


Colunas com valores faltantes:
quantidade        20
preco_unitario     2
vendedor          15
dtype: int64


In [24]:
# Versão visual: mostra todas as colunas com percentual de dados faltantes
percentual = (df.isnull().sum() / len(df) * 100).round(1)
resumo_faltantes = pd.DataFrame({
    'qtd_faltantes': df.isnull().sum(),
    'percentual (%)': percentual
})

resumo_faltantes[resumo_faltantes['qtd_faltantes'] > 0]


,qtd_faltantes,percentual (%)
quantidade,20,20.0
preco_unitario,2,2.0
vendedor,15,15.0


## 5. Problema de capitalização — coluna `produto`

O mesmo produto aparece escrito de formas diferentes: `"notebook"`, `"Notebook"`, `"NOTEBOOK"`.  
Para o pandas, esses são três produtos distintos — o que vai estragar qualquer análise de vendas por produto.

`df['produto'].unique()` mostra todos os valores únicos da coluna.

In [25]:
produtos_unicos = sorted(df['produto'].dropna().unique())

print(f"Total de valores únicos em 'produto': {len(produtos_unicos)}")
print()
for p in produtos_unicos:
    print(f"  '{p}'")


Total de valores únicos em 'produto': 18

  'CADEIRA GAMER'
  'Cadeira Gamer'
  'MONITOR'
  'MOUSE'
  'Monitor'
  'Mouse'
  'NOTEBOOK'
  'Notebook'
  'TECLADO'
  'Teclado'
  'WEBCAM'
  'Webcam'
  'cadeira gamer'
  'monitor'
  'mouse'
  'notebook'
  'teclado'
  'webcam'


In [26]:
# Consequência do problema: o pandas enxerga 18 "produtos" em vez de 6
# Se tentarmos contar vendas por produto agora, o resultado vai ser errado
print("Contagem de vendas por produto (ANTES da limpeza — resultado incorreto):")
print(df['produto'].value_counts())


Contagem de vendas por produto (ANTES da limpeza — resultado incorreto):
produto
NOTEBOOK         8
mouse            8
Webcam           7
TECLADO          7
Monitor          7
notebook         7
Notebook         6
Cadeira Gamer    6
MONITOR          6
teclado          6
cadeira gamer    5
MOUSE            4
webcam           4
CADEIRA GAMER    4
Mouse            4
monitor          4
WEBCAM           4
Teclado          3
Name: count, dtype: int64


## 6. Problema de formato — coluna `data_venda`

As datas estão em três formatos diferentes no mesmo arquivo:
- `2024-01-03` — formato ISO (ano-mês-dia)
- `05/01/2024` — formato brasileiro com barra (dia/mês/ano)
- `09-01-2024` — formato com hífen (dia-mês-ano)

Enquanto não padronizarmos, o pandas não consegue fazer operações com datas (ordenar, filtrar por mês, calcular períodos etc.).

In [27]:
datas_unicas = sorted(df['data_venda'].dropna().unique())

print(f"Total de datas únicas: {len(datas_unicas)}")
print()
for d in datas_unicas:
    print(f"  '{d}'")


Total de datas únicas: 100

  '01/02/2024'
  '01/03/2024'
  '02/04/2024'
  '03/09/2024'
  '03/10/2024'
  '04-08-2024'
  '05-02-2024'
  '05-06-2024'
  '05/01/2024'
  '05/07/2024'
  '06-03-2024'
  '06/05/2024'
  '07-04-2024'
  '09-01-2024'
  '09-09-2024'
  '10/02/2024'
  '10/08/2024'
  '11-07-2024'
  '11/06/2024'
  '12-05-2024'
  '12/01/2024'
  '12/03/2024'
  '13/04/2024'
  '14-02-2024'
  '15/09/2024'
  '16-08-2024'
  '17-03-2024'
  '17-06-2024'
  '17/07/2024'
  '18-01-2024'
  '18/05/2024'
  '19-04-2024'
  '20/02/2024'
  '2024-01-03'
  '2024-01-08'
  '2024-01-11'
  '2024-01-15'
  '2024-01-20'
  '2024-01-25'
  '2024-01-30'
  '2024-02-03'
  '2024-02-07'
  '2024-02-12'
  '2024-02-17'
  '2024-02-22'
  '2024-02-27'
  '2024-03-04'
  '2024-03-09'
  '2024-03-14'
  '2024-03-19'
  '2024-03-25'
  '2024-03-30'
  '2024-04-05'
  '2024-04-10'
  '2024-04-16'
  '2024-04-22'
  '2024-04-28'
  '2024-05-03'
  '2024-05-09'
  '2024-05-15'
  '2024-05-21'
  '2024-05-27'
  '2024-06-02'
  '2024-06-08'
  '2024-06-1

In [28]:
# Identifica qual formato cada data usa
import re

formato_iso    = []  # 2024-01-03
formato_barra  = []  # 05/01/2024
formato_hifen  = []  # 09-01-2024

for data in datas_unicas:
    if re.match(r'^\d{4}-\d{2}-\d{2}$', data):
        formato_iso.append(data)
    elif re.match(r'^\d{2}/\d{2}/\d{4}$', data):
        formato_barra.append(data)
    elif re.match(r'^\d{2}-\d{2}-\d{4}$', data):
        formato_hifen.append(data)

print(f"Formato ISO  (YYYY-MM-DD):  {len(formato_iso)} datas   → ex: {formato_iso[0]}")
print(f"Formato barra (DD/MM/YYYY): {len(formato_barra)} datas   → ex: {formato_barra[0]}")
print(f"Formato hífen (DD-MM-YYYY): {len(formato_hifen)} datas   → ex: {formato_hifen[0]}")


Formato ISO  (YYYY-MM-DD):  50 datas   → ex: 2024-01-03
Formato barra (DD/MM/YYYY): 26 datas   → ex: 01/02/2024
Formato hífen (DD-MM-YYYY): 24 datas   → ex: 04-08-2024


## 7. Problema de formato — coluna `preco_unitario`

Os preços usam vírgula como separador decimal e ponto como separador de milhar (padrão brasileiro).  
Ex: `"3.500,00"` em vez de `3500.00`.

O pandas não consegue converter esse formato diretamente para número — precisa de tratamento antes.

In [29]:
print("Valores únicos em 'preco_unitario':")
print(df['preco_unitario'].unique())
print()

# Tenta converter para número — vai falhar por causa da vírgula
try:
    pd.to_numeric(df['preco_unitario'])
except Exception as erro:
    print(f"Erro ao converter para número: {erro}")


Valores únicos em 'preco_unitario':
['3.500,00' '85,50' '220,00' '1.200,00' '890,00' nan '350,00']

Erro ao converter para número: Unable to parse string "3.500,00" at position 0


## 8. Resumo dos problemas encontrados

| Problema | Coluna(s) | Impacto |
|---|---|---|
| Valores faltantes | `quantidade`, `vendedor`, `preco_unitario` | Cálculos de total ficam errados ou incompletos |
| Capitalização inconsistente | `produto` | Agrupa produtos iguais como diferentes |
| Formatos de data misturados | `data_venda` | Não consegue ordenar nem filtrar por período |
| Separador decimal com vírgula | `preco_unitario` | Não consegue fazer cálculos numéricos |

**Próximo passo:** limpar cada um desses problemas nas seções abaixo.

---

# Limpeza dos Dados

Agora que sabemos exatamente quais são os problemas, vamos corrigi-los um por um.  
Todas as alterações são feitas em cima do `df` original — o arquivo `vendas.csv` nunca é tocado.

## 9. Padronizar a coluna `produto` (title case)

**Title case** significa capitalizar a primeira letra de cada palavra: `"cadeira gamer"` → `"Cadeira Gamer"`.  
O método `.str.title()` do pandas faz isso em uma linha para a coluna inteira.

In [30]:
# .str acessa métodos de texto para colunas do pandas
# .title() transforma a string para title case: primeira letra de cada palavra em maiúscula
df['produto'] = df['produto'].str.title()

# Verifica o resultado: agora deve haver apenas 6 produtos únicos
print("Produtos únicos após padronização:")
print(sorted(df['produto'].dropna().unique()))
print()
print("Contagem de vendas por produto (agora correta):")
print(df['produto'].value_counts())


Produtos únicos após padronização:
['Cadeira Gamer', 'Monitor', 'Mouse', 'Notebook', 'Teclado', 'Webcam']

Contagem de vendas por produto (agora correta):
produto
Notebook         21
Monitor          17
Mouse            16
Teclado          16
Cadeira Gamer    15
Webcam           15
Name: count, dtype: int64


## 10. Unificar os formatos de data para `YYYY-MM-DD`

O pandas tem a função `pd.to_datetime()` que converte strings para datas.  
O problema aqui é que temos **três formatos diferentes no mesmo arquivo**, então não podemos passar um único formato.

A solução é processar cada formato separadamente e depois juntar tudo:
1. Datas ISO (`2024-01-03`) já estão no formato certo — só converter
2. Datas com barra (`05/01/2024`) — informamos o formato `%d/%m/%Y`
3. Datas com hífen (`09-01-2024`) — informamos o formato `%d-%m-%Y`

In [31]:
def converter_data(data):
    """Recebe uma string de data em qualquer um dos três formatos e retorna um objeto datetime."""
    if pd.isnull(data):
        return pd.NaT  # NaT é o equivalente de NaN para datas

    # Tenta cada formato em ordem — o primeiro que funcionar é retornado
    for formato in ('%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y'):
        try:
            return pd.to_datetime(data, format=formato)
        except ValueError:
            continue  # esse formato não funcionou, tenta o próximo

    # Se nenhum formato funcionou, retorna NaT para não quebrar o código
    return pd.NaT

# Aplica a função em cada linha da coluna data_venda
# .apply() percorre a coluna linha por linha e chama a função em cada valor
df['data_venda'] = df['data_venda'].apply(converter_data)

# Verifica o resultado
print(f"Tipo da coluna depois da conversão: {df['data_venda'].dtype}")
print()
print("Primeiras datas convertidas:")
print(df['data_venda'].head(10))


Tipo da coluna depois da conversão: datetime64[ns]

Primeiras datas convertidas:
0   2024-01-03
1   2024-01-05
2   2024-01-08
3   2024-01-09
4   2024-01-11
5   2024-01-12
6   2024-01-15
7   2024-01-18
8   2024-01-20
9   2024-01-22
Name: data_venda, dtype: datetime64[ns]


## 11. Converter `preco_unitario` de string com vírgula para float

Os preços estão no formato brasileiro: `"3.500,00"` (ponto = milhar, vírgula = decimal).  
Para o Python, o formato numérico correto é `3500.00` (sem ponto de milhar, ponto = decimal).

Precisamos fazer dois ajustes na string antes de converter:
1. Remover o ponto de milhar: `"3.500,00"` → `"3500,00"`
2. Trocar a vírgula pelo ponto: `"3500,00"` → `"3500.00"`

Depois disso, `pd.to_numeric()` consegue converter normalmente.

In [32]:
# Passo 1: remove o ponto de milhar  → "3.500,00" vira "3500,00"
# Passo 2: troca a vírgula pelo ponto → "3500,00"  vira "3500.00"
# .str.replace() substitui texto dentro de cada célula da coluna
df['preco_unitario'] = (
    df['preco_unitario']
    .str.replace('.', '', regex=False)   # remove o ponto (regex=False = trata como texto simples)
    .str.replace(',', '.', regex=False)  # troca vírgula por ponto
)

# Agora converte para número de ponto flutuante (float)
# errors='coerce' transforma qualquer valor que não conseguir converter em NaN
# (em vez de quebrar com erro)
df['preco_unitario'] = pd.to_numeric(df['preco_unitario'], errors='coerce')

# Verifica o resultado
print(f"Tipo da coluna depois da conversão: {df['preco_unitario'].dtype}")
print()
print("Valores únicos de preco_unitario:")
print(sorted(df['preco_unitario'].dropna().unique()))


Tipo da coluna depois da conversão: float64

Valores únicos de preco_unitario:
[np.float64(85.5), np.float64(220.0), np.float64(350.0), np.float64(890.0), np.float64(1200.0), np.float64(3500.0)]


## 12. Tratar valores faltantes

Temos três situações distintas que pedem estratégias diferentes:

- **`quantidade`** — valor numérico. Usar a **mediana do produto** é mais inteligente do que a mediana geral, porque produtos diferentes têm padrões de compra diferentes (ninguém compra 2 notebooks de uma vez, mas pode comprar 5 mouses).
- **`preco_unitario`** — também numérico, com a mesma lógica: cada produto tem seu próprio preço, então faz mais sentido usar a mediana por produto do que um valor geral.
- **`vendedor`** — não tem como estimar quem fez a venda. A opção honesta é preencher com `"Não informado"` para deixar claro que o dado está ausente.

### O que é mediana?
A mediana é o valor do meio quando você ordena todos os números.  
Ela é preferível à média quando existem valores muito discrepantes (ex: uma venda de 10 notebooks distorceria a média).


In [33]:
# Antes de preencher: quantidade ainda está como string — precisa converter para número
df['quantidade'] = pd.to_numeric(df['quantidade'], errors='coerce')

print("Valores faltantes ANTES do preenchimento:")
print(df[['quantidade', 'preco_unitario', 'vendedor']].isnull().sum())


Valores faltantes ANTES do preenchimento:
quantidade        20
preco_unitario     2
vendedor          15
dtype: int64


In [34]:
# --- quantidade: preenche com a mediana do produto correspondente ---

# groupby('produto') agrupa as linhas por produto
# transform('median') calcula a mediana de cada grupo e devolve uma Series
# com o mesmo tamanho do df original — cada linha recebe a mediana do seu produto
mediana_quantidade_por_produto = df.groupby('produto')['quantidade'].transform('median')

# fillna() preenche apenas os NaN, mantendo os valores existentes
# .astype(int) converte para inteiro — só é seguro fazer isso depois do fillna(),
# porque int não aceita NaN (por isso a coluna vira float quando há nulos)
df['quantidade'] = df['quantidade'].fillna(mediana_quantidade_por_produto).astype(int)

# --- preco_unitario: mesma lógica ---
mediana_preco_por_produto = df.groupby('produto')['preco_unitario'].transform('median')
df['preco_unitario'] = df['preco_unitario'].fillna(mediana_preco_por_produto)

# --- vendedor: preenche com texto fixo ---
df['vendedor'] = df['vendedor'].fillna('Não informado')

print("Valores faltantes DEPOIS do preenchimento:")
print(df[['quantidade', 'preco_unitario', 'vendedor']].isnull().sum())
print()
print(f"Tipo da coluna quantidade: {df['quantidade'].dtype}")
print(df['quantidade'].head(10))


Valores faltantes DEPOIS do preenchimento:
quantidade        0
preco_unitario    0
vendedor          0
dtype: int64

Tipo da coluna quantidade: int64
0    1
1    2
2    1
3    1
4    2
5    1
6    3
7    2
8    1
9    1
Name: quantidade, dtype: int64


In [35]:
# Confere quais linhas tinham vendedor faltante — agora com "Não informado"
print("Linhas onde vendedor foi preenchido automaticamente:")
df[df['vendedor'] == 'Não informado'][['id_venda', 'produto', 'vendedor']]


Linhas onde vendedor foi preenchido automaticamente:


,id_venda,produto,vendedor
4,5,Notebook,Não informado
11,12,Webcam,Não informado
17,18,Cadeira Gamer,Não informado
24,25,Notebook,Não informado
31,32,Monitor,Não informado
37,38,Monitor,Não informado
44,45,Teclado,Não informado
51,52,Monitor,Não informado
58,59,Mouse,Não informado
65,66,Webcam,Não informado


## 13. Verificação final antes de salvar

Antes de gravar o arquivo limpo, confere o estado do `df` como um todo.

In [36]:
print("=== Visão geral do dataframe limpo ===")
print()
df.info()


=== Visão geral do dataframe limpo ===

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_venda        100 non-null    object        
 1   data_venda      100 non-null    datetime64[ns]
 2   produto         100 non-null    object        
 3   categoria       100 non-null    object        
 4   quantidade      100 non-null    int64         
 5   preco_unitario  100 non-null    float64       
 6   vendedor        100 non-null    object        
 7   regiao          100 non-null    object        
 8   status          100 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(6)
memory usage: 7.2+ KB


In [37]:
print("Valores faltantes restantes:")
print(df.isnull().sum())
print()
print("Primeiras linhas do dataframe limpo:")
df.head(10)


Valores faltantes restantes:
id_venda          0
data_venda        0
produto           0
categoria         0
quantidade        0
preco_unitario    0
vendedor          0
regiao            0
status            0
dtype: int64

Primeiras linhas do dataframe limpo:


,id_venda,data_venda,produto,categoria,quantidade,preco_unitario,vendedor,regiao,status
0,1,2024-01-03,Notebook,Eletrônicos,1,3500.0,Carlos Silva,Sudeste,concluída
1,2,2024-01-05,Mouse,Periféricos,2,85.5,Ana Lima,Sul,concluída
2,3,2024-01-08,Teclado,Periféricos,1,220.0,João Costa,Nordeste,concluída
3,4,2024-01-09,Monitor,Eletrônicos,1,1200.0,Carlos Silva,Sudeste,concluída
4,5,2024-01-11,Notebook,Eletrônicos,2,3500.0,Não informado,Sul,concluída
5,6,2024-01-12,Cadeira Gamer,Móveis,1,890.0,Ana Lima,Sudeste,cancelada
6,7,2024-01-15,Mouse,Periféricos,3,85.5,Pedro Rocha,Centro-Oeste,concluída
7,8,2024-01-18,Teclado,Periféricos,2,220.0,Carlos Silva,Nordeste,pendente
8,9,2024-01-20,Monitor,Eletrônicos,1,1200.0,João Costa,Sul,concluída
9,10,2024-01-22,Notebook,Eletrônicos,1,3500.0,Ana Lima,Sudeste,concluída


## 14. Salvar o arquivo limpo

Salva o resultado em `dados/vendas_limpo.csv`.  
O arquivo original `dados/vendas.csv` permanece intacto — boa prática em projetos de dados.

In [38]:
# index=False evita que o pandas escreva a coluna de índice (0, 1, 2...) no CSV
# date_format='%Y-%m-%d' garante que as datas sejam salvas no formato ISO, sem hora
df.to_csv('../dados/vendas_limpo.csv', index=False, date_format='%Y-%m-%d')

print("Arquivo salvo em: dados/vendas_limpo.csv")
print(f"Linhas salvas: {len(df)}")


Arquivo salvo em: dados/vendas_limpo.csv
Linhas salvas: 100
